In [ ]:
import os

def build_depth_map(super_path):
    depth_map_path = {}

    # Walk through the directory tree
    for root, _, files in os.walk(super_path):
        for filename in files:
            if filename.endswith('.png'):
                full_path = os.path.join(root, filename)
                # Remove super_path from the full path to get the relative path
                relative_path = os.path.relpath(full_path, super_path)
                # Split the relative path into parts (folders)
                parts = relative_path.split(os.sep)
                parts[-1]=parts[-1][len('result_merged_depth_'):-len('.png')]
                
                # Initialize nested dictionaries as needed
                current_dict = depth_map_path
                for part in parts[:-1]:  # Iterate over all parts except the last one (filename)
                    current_dict = current_dict.setdefault(part, {})
                
                # Assign the full path to the deepest nested dictionary
                current_dict[parts[-1]] = full_path

    return depth_map_path

def build_depth_map_confidence(super_path):
    depth_map_path = {}

    # Walk through the directory tree
    for root, _, files in os.walk(super_path):
        for filename in files:
            if filename.endswith('.exr'):
                full_path = os.path.join(root, filename)
                # Remove super_path from the full path to get the relative path
                relative_path = os.path.relpath(full_path, super_path)
                # Split the relative path into parts (folders)
                parts = relative_path.split(os.sep)
                parts[-1]=parts[-1][len('result_merged_conf_'):-len('.exr')]
                
                # Initialize nested dictionaries as needed
                current_dict = depth_map_path
                for part in parts[:-1]:  # Iterate over all parts except the last one (filename)
                    current_dict = current_dict.setdefault(part, {})
                
                # Assign the full path to the deepest nested dictionary
                current_dict[parts[-1]] = full_path

    return depth_map_path



In [ ]:
import os

def find_train_folders(directory):
    train_folders = []
    for root, dirs, files in os.walk(directory):
        for dir in dirs:
            if dir.startswith('train'):
                train_folders.append(os.path.join(root, dir))
    return train_folders

train_path=find_train_folders('/mnt/Velocity Vault/Personal/Projects/Python/Autofocus/Train/')

depth_map_path={}
depth_map_path_temp={}

for path in train_path:
    depth_map_path_temp=build_depth_map(path+"/merged_depth")
    depth_map_path.update(depth_map_path_temp)

depth_map_confidence_path={}
depth_map_confidence_path_temp={}

for path in train_path:
    depth_map_confidence_path_temp=build_depth_map_confidence(path+"/merged_conf")
    depth_map_confidence_path.update(depth_map_confidence_path_temp)




import pprint
pprint.pprint(depth_map_path)
pprint.pprint(depth_map_confidence_path)


In [ ]:
import numpy as np
import random

def generate_random_patches(size=(504,378)):
    image=np.zeros(size)
    patches=[]
    for _ in (range(100)):
        x=random.randint(16, 487)
        y=random.randint(16, 361)

        #print(x,y)
        
        breaker=False

        for i in range(x-26,x+26):
            for j in range(y-26,y+26):
                if i<0 or i>503 or j<0 or j>377 :
                    continue
                if image[i][j]==1:
                    breaker=True
                    break
            if breaker:
                break

        if breaker:
            continue
        
        x=x-16
        y=y-16

        for i in range(x,x+33):
            for j in range(y,y+33):
                image[i][j]=1

        patches.append((x,y))

    return patches

In [ ]:
import numpy as np
import cv2
import OpenEXR
import Imath

def load_exr(file_path):
    exr_file = OpenEXR.InputFile(file_path)
    header = exr_file.header()
    dw = header['dataWindow']
    width = dw.max.x - dw.min.x + 1
    height = dw.max.y - dw.min.y + 1
    pt = Imath.PixelType(Imath.PixelType.FLOAT)
    r = np.frombuffer(exr_file.channel('R', pt), dtype=np.float32)
    r.shape = (height, width)
    return r

def predict_focal_length(depth_map_path,depth_confidence_path):
    depth_map = cv2.imread(depth_map_path, cv2.IMREAD_GRAYSCALE).astype(np.float32)
    depth_map_confidence = load_exr(depth_confidence_path)

    patches=generate_random_patches()

    confidence_patch_blocks=[depth_map_confidence[x:x+32,y:y+32] for (x,y) in patches]
    median_confidence_patch_block=[np.median(patch.flatten()) for patch in confidence_patch_blocks]
    patch_index=median_confidence_patch_block.index(max(median_confidence_patch_block))
    patch=patches[patch_index]

    depth_values=depth_map[patch[0]:patch[0]+32,patch[1]:patch[1]+32]
    depth_values=depth_values.flatten()

    # Define max and min values
    max_depth = 100.0
    min_depth = 0.2

    depth_map_in_meters = (max_depth * min_depth) / (max_depth - (max_depth - min_depth) * (depth_values / 255.0))

    # Compute the median value in the entire depth map
    median_depth = np.median(depth_map_in_meters)

    final_focus=median_depth*1000

    return (final_focus,patch)

slice_focal_length=[3910.92,2289.27,1508.71,1185.83,935.91,801.09,700.37,605.39,546.23,486.87,447.99,407.40,379.91,350.41,329.95,307.54,291.72,274.13,261.53,247.35,237.08,225.41,216.88,207.10,198.18,191.60,183.96,178.29,171.69,165.57,160.99,155.61,150.59,146.81,142.35,138.98,134.99,131.23,127.69,124.99,121.77,118.73,116.40,113.63,110.99,108.47,106.54,104.23,102.01]

def find_closest(value, num_list):
    closest_value = min(num_list, key=lambda x: abs(x - value))
    return closest_value

def predict_slice(depth_map_path,depth_confidence_path):
    output=predict_focal_length(depth_map_path,depth_confidence_path)
    predicted_focus=output[0]
    patch=output[1]
    closest_value = find_closest(predicted_focus, slice_focal_length)
    true_slice=slice_focal_length.index(closest_value)
    return (true_slice,patch)

In [ ]:
from tqdm import tqdm
import copy

ground_truth=copy.deepcopy(depth_map_path)
patches=copy.deepcopy(depth_map_path)

for image_type in tqdm(ground_truth):
    for pos in ground_truth[image_type]:
        output=predict_slice(depth_map_path[image_type][pos],depth_map_confidence_path[image_type][pos])
        ground_truth[image_type][pos]=output[0]
        #print(ground_truth[image_type][pos])
        patches[image_type][pos]=output[1]

import pprint
pprint.pprint(ground_truth)
pprint.pprint(patches)

In [ ]:
import os

def generate_image_paths(super_path):
    # Initialize the dictionary
    image_path = {}

    # Traverse the directory structure
    for root, dirs, files in os.walk(super_path):
        for file in files:
            if file.endswith('.png'):
                # Get the relative path from super_path
                relative_path = os.path.relpath(root, super_path)
                # Split the relative path to get <string> and <integer>
                string_part, integer_part = os.path.split(relative_path)
                integer_part = int(integer_part)
                
                # Extract the position from the filename
                file_name_parts = file.split('_')
                position = file_name_parts[-1].split('.')[0]  # Extract 'bottom', 'top', etc.
                
                # Construct the full file path
                full_file_path = os.path.join(root, file)
                
                # Populate the dictionary
                if string_part not in image_path:
                    image_path[string_part] = {}
                if integer_part not in image_path[string_part]:
                    image_path[string_part][integer_part] = {}
                
                image_path[string_part][integer_part][position] = full_file_path

    return image_path



In [ ]:

left_image_path = {}
left_image_path_temp = {}
right_image_path = {}
right_image_path_temp = {}


def find_train_folders(directory):
    train_folders = []
    for root, dirs, files in os.walk(directory):
        for dir in dirs:
            if dir.startswith('train'):
                train_folders.append(os.path.join(root, dir))
    return train_folders


train_path = find_train_folders(
    '/mnt/Velocity Vault/Personal/Projects/Python/Autofocus/Train/')


for path in train_path:
    left_image_path_temp = generate_image_paths(path+'/raw_up_left_pd')
    left_image_path.update(left_image_path_temp)
    right_image_path_temp = generate_image_paths(path+'/raw_up_right_pd')
    right_image_path.update(right_image_path_temp)


import pprint

pprint.pprint(left_image_path)
pprint.pprint(right_image_path)
print(len(left_image_path))
print(len(right_image_path))


In [ ]:
def build_dataset_labels(image_path,ground_truth,patches,count=0,dataset=[],labels=[],crops=[]):
    for image_type in image_path:
        if count==len(dataset):

            crops.append([])
            crops.append([])
            crops.append([])
            crops.append([])
            crops.append([])

            labels.append([])
            labels.append([])
            labels.append([])
            labels.append([])
            labels.append([])

            dataset.append([])
            dataset.append([])
            dataset.append([])
            dataset.append([])
            dataset.append([])

        for focal_slice in image_path[image_type]:
            dataset[count].append(image_path[image_type][focal_slice]['top'])
            dataset[count+1].append(image_path[image_type][focal_slice]['bottom'])
            dataset[count+2].append(image_path[image_type][focal_slice]['center'])
            dataset[count+3].append(image_path[image_type][focal_slice]['left'])
            dataset[count+4].append(image_path[image_type][focal_slice]['right'])
        
        crops[count]=patches[image_type]['top']
        crops[count+1]=patches[image_type]['bottom']
        crops[count+2]=patches[image_type]['center']
        crops[count+3]=patches[image_type]['left']
        crops[count+4]=patches[image_type]['right']

        labels[count]=ground_truth[image_type]['top']
        labels[count+1]=ground_truth[image_type]['bottom']
        labels[count+2]=ground_truth[image_type]['center']
        labels[count+3]=ground_truth[image_type]['left']
        labels[count+4]=ground_truth[image_type]['right']

        count+=5
    return dataset,labels,crops


dataset_path,labels,crops=build_dataset_labels(left_image_path,ground_truth,patches)
dataset_path,_,_=build_dataset_labels(right_image_path,ground_truth,patches,dataset=dataset_path)
crops=[(x*4,y*4) for (x,y) in crops]

pprint.pprint(dataset_path)
pprint.pprint(labels)
pprint.pprint(crops)
print(len(dataset_path))
print(len(labels))
print(len(crops))

In [ ]:
def load_images(dataset_path,crops):
    dataset=copy.deepcopy(dataset_path)
    for image_set in tqdm(range(len(dataset_path))):
        for focal_slice in range(len(dataset_path[image_set])):

            image=cv2.imread(dataset_path[image_set][focal_slice])

            x=crops[image_set][0]
            y=crops[image_set][1]
            image_patch=image[x:x+128,y:y+128]

            dataset[image_set][focal_slice]=image_patch

    return dataset

dataset=load_images(dataset_path,crops)
pprint.pprint(dataset)